# Previsão de Retorno para Revisão - Honda

**FGV - Modelagem Preditiva - Entrega 22/10/2026**

**Grupo:** João Victor Fortunato Teixeira Mundim, Martinho Aloisio Lenz, Nicolas Henrique Richardt da Silva, Renato Vargas Monteiro, Rodrigo Pimentel Carneiro

## Objetivo
Prever se um cliente que comprou uma motocicleta Honda irá retornar para revisão.

## Modelos
1. Árvore de Decisão (CART/Entropia)
2. Regressão Logística
3. Random Forest
4. Gradient Boosting

**Nota:** Pipeline corrigido - pré-processamento aplicado separadamente em treino/teste para evitar vazamento de dados.

In [ ]:
# 1. IMPORTAÇÃO DAS BIBLIOTECAS
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, 
                             roc_auc_score, roc_curve, precision_score, recall_score, f1_score)

import warnings
warnings.filterwarnings('ignore')

print("Bibliotecas importadas com sucesso!")

In [ ]:
# 2. CARREGAMENTO DOS DADOS
# Para Google Colab:
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/Honda_Revisao_Modelagem.csv')

# Para execução local:
df = pd.read_csv('Honda_Revisao_Modelagem.csv')

print("="*60)
print("DATASET: PREVISÃO DE RETORNO PARA REVISÃO - HONDA")
print("="*60)
print(f"\nRegistros: {df.shape[0]}")
print(f"Variáveis: {df.shape[1]}")
print(f"\nColunas: {list(df.columns)}")
print(f"\nTaxa de retorno: {df['RETORNOU_REVISAO'].mean()*100:.1f}%")
df.head()

In [ ]:
# 3. PREPARAÇÃO DOS DADOS
print("="*60)
print("PREPARAÇÃO DOS DADOS")
print("="*60)

X = df.drop(columns=['RETORNOU_REVISAO']).copy()
y = df['RETORNOU_REVISAO'].copy()

colunas_cat = X.select_dtypes(include=['object']).columns.tolist()
colunas_num = X.select_dtypes(include=['number']).columns.tolist()

print(f"\nVariáveis Categóricas ({len(colunas_cat)}): {colunas_cat}")
print(f"\nVariáveis Numéricas ({len(colunas_num)}): {colunas_num}")

In [ ]:
# 4. DIVISÃO TREINO/TESTE (ANTES DO PRÉ-PROCESSAMENTO)
print("="*60)
print("DIVISÃO TREINO/TESTE")
print("="*60)

random_state = 42

# IMPORTANTE: Dividir ANTES de qualquer transformação
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=random_state, stratify=y
)

print(f"\nDados de Treino: {len(y_train)} registros ({len(y_train)/len(y)*100:.0f}%)")
print(f"Dados de Teste:  {len(y_test)} registros ({len(y_test)/len(y)*100:.0f}%)")
print(f"\nBaseline (classe majoritária): {(y_test==0).mean()*100:.2f}%")

In [ ]:
# 5. PRÉ-PROCESSAMENTO (FIT APENAS NO TREINO)
print("="*60)
print("PRÉ-PROCESSAMENTO (SEM VAZAMENTO DE DADOS)")
print("="*60)

# Label Encoding para modelos de árvore
X_train_tree = X_train.copy()
X_test_tree = X_test.copy()
encoders = {}

print("\nLabel Encoding (fit apenas no treino):")
for col in colunas_cat:
    le = LabelEncoder()
    X_train_tree[col] = le.fit_transform(X_train[col].astype(str))
    X_test_tree[col] = X_test[col].astype(str).apply(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )
    encoders[col] = le
    print(f"   {col}: {len(le.classes_)} classes")

# One-Hot + Scaler para Regressão Logística
print("\nOne-Hot Encoding + StandardScaler (fit apenas no treino):")
X_train_log = pd.get_dummies(X_train, columns=colunas_cat, drop_first=True)
X_test_log = pd.get_dummies(X_test, columns=colunas_cat, drop_first=True)

for col in X_train_log.columns:
    if col not in X_test_log.columns:
        X_test_log[col] = 0
X_test_log = X_test_log[X_train_log.columns]

scaler = StandardScaler()
X_train_log_scaled = pd.DataFrame(scaler.fit_transform(X_train_log), columns=X_train_log.columns)
X_test_log_scaled = pd.DataFrame(scaler.transform(X_test_log), columns=X_test_log.columns)

print(f"   Features após One-Hot: {X_train_log_scaled.shape[1]}")
print("\n✅ Pré-processamento concluído (sem vazamento)!")

In [ ]:
# 6. MODELO 1: ÁRVORE DE DECISÃO
print("="*60)
print("MODELO 1: ÁRVORE DE DECISÃO (CART/ENTROPIA)")
print("="*60)

arvore = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=5,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=random_state
)

arvore.fit(X_train_tree, y_train)
y_pred_arvore = arvore.predict(X_test_tree)
y_proba_arvore = arvore.predict_proba(X_test_tree)[:, 1]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
cv_arvore = cross_val_score(arvore, X_train_tree, y_train, cv=cv, scoring='accuracy')

print(f"\nResultados:")
print(f"   Acurácia Teste:  {accuracy_score(y_test, y_pred_arvore)*100:.2f}%")
print(f"   AUC-ROC:         {roc_auc_score(y_test, y_proba_arvore):.4f}")
print(f"   Precisão(1):     {precision_score(y_test, y_pred_arvore)*100:.2f}%")
print(f"   Recall(1):       {recall_score(y_test, y_pred_arvore)*100:.2f}%")
print(f"   CV Treino:       {cv_arvore.mean()*100:.2f}% (+/- {cv_arvore.std()*100:.2f}%)")

In [ ]:
# 7. MODELO 2: REGRESSÃO LOGÍSTICA
print("="*60)
print("MODELO 2: REGRESSÃO LOGÍSTICA")
print("="*60)

logistica = LogisticRegression(
    penalty='l2',
    C=1.0,
    solver='lbfgs',
    max_iter=1000,
    random_state=random_state
)

logistica.fit(X_train_log_scaled, y_train)
y_pred_log = logistica.predict(X_test_log_scaled)
y_proba_log = logistica.predict_proba(X_test_log_scaled)[:, 1]
cv_log = cross_val_score(logistica, X_train_log_scaled, y_train, cv=cv, scoring='accuracy')

print(f"\nResultados:")
print(f"   Acurácia Teste:  {accuracy_score(y_test, y_pred_log)*100:.2f}%")
print(f"   AUC-ROC:         {roc_auc_score(y_test, y_proba_log):.4f}")
print(f"   Precisão(1):     {precision_score(y_test, y_pred_log)*100:.2f}%")
print(f"   Recall(1):       {recall_score(y_test, y_pred_log)*100:.2f}%")
print(f"   CV Treino:       {cv_log.mean()*100:.2f}% (+/- {cv_log.std()*100:.2f}%)")

In [ ]:
# 8. MODELO 3: RANDOM FOREST
print("="*60)
print("MODELO 3: RANDOM FOREST")
print("="*60)

rf = RandomForestClassifier(
    n_estimators=100,
    criterion='entropy',
    max_depth=10,
    max_features='sqrt',
    random_state=random_state,
    n_jobs=-1
)

rf.fit(X_train_tree, y_train)
y_pred_rf = rf.predict(X_test_tree)
y_proba_rf = rf.predict_proba(X_test_tree)[:, 1]
cv_rf = cross_val_score(rf, X_train_tree, y_train, cv=cv, scoring='accuracy')

print(f"\nResultados:")
print(f"   Acurácia Teste:  {accuracy_score(y_test, y_pred_rf)*100:.2f}%")
print(f"   AUC-ROC:         {roc_auc_score(y_test, y_proba_rf):.4f}")
print(f"   Precisão(1):     {precision_score(y_test, y_pred_rf)*100:.2f}%")
print(f"   Recall(1):       {recall_score(y_test, y_pred_rf)*100:.2f}%")
print(f"   CV Treino:       {cv_rf.mean()*100:.2f}% (+/- {cv_rf.std()*100:.2f}%)")

In [ ]:
# 9. MODELO 4: GRADIENT BOOSTING
print("="*60)
print("MODELO 4: GRADIENT BOOSTING")
print("="*60)

gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    random_state=random_state
)

gb.fit(X_train_tree, y_train)
y_pred_gb = gb.predict(X_test_tree)
y_proba_gb = gb.predict_proba(X_test_tree)[:, 1]
cv_gb = cross_val_score(gb, X_train_tree, y_train, cv=cv, scoring='accuracy')

print(f"\nResultados:")
print(f"   Acurácia Teste:  {accuracy_score(y_test, y_pred_gb)*100:.2f}%")
print(f"   AUC-ROC:         {roc_auc_score(y_test, y_proba_gb):.4f}")
print(f"   Precisão(1):     {precision_score(y_test, y_pred_gb)*100:.2f}%")
print(f"   Recall(1):       {recall_score(y_test, y_pred_gb)*100:.2f}%")
print(f"   CV Treino:       {cv_gb.mean()*100:.2f}% (+/- {cv_gb.std()*100:.2f}%)")

In [ ]:
# 10. QUADRO COMPARATIVO
print("="*60)
print("QUADRO COMPARATIVO DE DESEMPENHO")
print("="*60)

# Armazenar resultados com mapeamento correto
resultados_dict = {
    'Árvore de Decisão': (y_pred_arvore, y_proba_arvore, cv_arvore),
    'Regressão Logística': (y_pred_log, y_proba_log, cv_log),
    'Random Forest': (y_pred_rf, y_proba_rf, cv_rf),
    'Gradient Boosting': (y_pred_gb, y_proba_gb, cv_gb)
}

resultados = []
for nome, (y_pred, y_proba, cv_scores) in resultados_dict.items():
    resultados.append({
        'Modelo': nome,
        'Acurácia': accuracy_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_proba),
        'Precisão(1)': precision_score(y_test, y_pred),
        'Recall(1)': recall_score(y_test, y_pred),
        'F1(1)': f1_score(y_test, y_pred),
        'CV Média': cv_scores.mean(),
        'CV Std': cv_scores.std()
    })

df_resultados = pd.DataFrame(resultados).sort_values('AUC-ROC', ascending=False)
print(f"\nBaseline (sempre prever 0): {(y_test==0).mean()*100:.2f}%\n")
print(df_resultados.to_string(index=False))

In [ ]:
# 11. VISUALIZAÇÕES
print("="*60)
print("VISUALIZAÇÕES")
print("="*60)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Curvas ROC
cores = {'Árvore de Decisão': 'blue', 'Regressão Logística': 'purple', 
         'Random Forest': 'orange', 'Gradient Boosting': 'green'}

for nome, (y_pred, y_proba, cv_scores) in resultados_dict.items():
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    axes[0,0].plot(fpr, tpr, color=cores[nome], lw=2, label=f'{nome} (AUC={auc:.3f})')

axes[0,0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0,0].set_xlabel('Taxa de Falso Positivo')
axes[0,0].set_ylabel('Taxa de Verdadeiro Positivo')
axes[0,0].set_title('Curvas ROC Comparativas')
axes[0,0].legend(loc='lower right')
axes[0,0].grid(True, alpha=0.3)

# 2. Métricas comparativas
modelos = list(resultados_dict.keys())
metricas = ['Acurácia', 'Precisão(1)', 'Recall(1)', 'F1(1)']
x = np.arange(len(modelos))
width = 0.2

for i, metrica in enumerate(metricas):
    valores = [r[metrica] for r in resultados]
    axes[0,1].bar(x + i*width, valores, width, label=metrica)

axes[0,1].set_xticks(x + width*1.5)
axes[0,1].set_xticklabels(modelos, rotation=45, ha='right')
axes[0,1].set_ylabel('Valor')
axes[0,1].set_title('Comparação de Métricas')
axes[0,1].legend()
axes[0,1].set_ylim(0, 1)

# 3. Matriz de Confusão do MELHOR modelo (por AUC)
melhor = df_resultados.iloc[0]['Modelo']
melhor_pred = resultados_dict[melhor][0]
cm = confusion_matrix(y_test, melhor_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1,0],
            xticklabels=['Não Retornou', 'Retornou'],
            yticklabels=['Não Retornou', 'Retornou'])
axes[1,0].set_title(f'Matriz de Confusão - {melhor}')
axes[1,0].set_xlabel('Predito')
axes[1,0].set_ylabel('Real')

# 4. Feature Importance (RF)
importancias = pd.DataFrame({
    'Feature': X_train_tree.columns,
    'Importância': rf.feature_importances_
}).sort_values('Importância', ascending=True).tail(10)

axes[1,1].barh(range(len(importancias)), importancias['Importância'], color='steelblue')
axes[1,1].set_yticks(range(len(importancias)))
axes[1,1].set_yticklabels(importancias['Feature'])
axes[1,1].set_xlabel('Importância')
axes[1,1].set_title('Top 10 Features (Random Forest)')

plt.tight_layout()
plt.savefig('graficos_comparativos.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 12. CONCLUSÃO
print("="*70)
print("CONCLUSÃO")
print("="*70)

melhor = df_resultados.iloc[0]

print(f"\n🏆 MELHOR MODELO: {melhor['Modelo']}")
print(f"\n📊 Desempenho:")
print(f"   • Acurácia: {melhor['Acurácia']*100:.2f}%")
print(f"   • AUC-ROC: {melhor['AUC-ROC']:.4f}")
print(f"   • Precisão(1): {melhor['Precisão(1)']*100:.2f}%")
print(f"   • Recall(1): {melhor['Recall(1)']*100:.2f}%")
print(f"   • CV Treino: {melhor['CV Média']*100:.2f}% (+/- {melhor['CV Std']*100:.2f}%)")

print(f"\n📋 INTERPRETAÇÃO DAS MÉTRICAS:")
print(f"   • Recall(1) = {melhor['Recall(1)']*100:.2f}% dos clientes que RETORNARAM foram identificados")
print(f"   • Para identificar clientes em RISCO de não retornar:")
print(f"     usar probabilidade_nao_retorno = 1 - probabilidade_retorno")

print("\n" + "="*70)